In [ ]:
# This chunk of code can be used residue wise contact map between protein residues and DNA bases. The code finally give contact map of protein-DNA
# for each window and each combination of tail and DNA.

In [ ]:
import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
from MDAnalysis.lib.distances import capped_distance

top = "nucl.pdb"
cutoff = 4.5
umbrellas = range(12)

for window in umbrellas:
    traj = f"window{window}.pbc.xtc"
    u = mda.Universe(top, traj)

    # H3 N-terminal tails: residues 1-44, heavy atoms
    h3_tail1 = u.select_atoms(
        "segid C and resid 1:44 and not name H*"
    )
    h3_tail2 = u.select_atoms(
        "segid G and resid 1:44 and not name H*"
    )

    # DNA ends: 30 bp from both strands, heavy atoms
    dna_end1 = u.select_atoms(
        "((chainID A and resid 116:145) or "
        "(chainID B and resid 1:30)) and not name H*"
    )

    dna_end2 = u.select_atoms(
        "((chainID A and resid 1:30) or "
        "(chainID B and resid 116:145)) and not name H*"
    )

    tail1_residues = h3_tail1.residues
    tail2_residues = h3_tail2.residues
    dna1_residues = dna_end1.residues
    dna2_residues = dna_end2.residues

    print(
        f"Umbrella {window}\n"
        f"Tail 1 residues: {len(tail1_residues)}, "
        f"DNA end 1 nucleotides: {len(dna1_residues)}\n"
        f"Tail 2 residues: {len(tail2_residues)}, "
        f"DNA end 2 nucleotides: {len(dna2_residues)}"
    )

    # Residue-level contact-frequency matrices
    contact_map1 = np.zeros(
        (len(tail1_residues), len(dna1_residues)),
        dtype=float
    )
    contact_map2 = np.zeros(
        (len(tail2_residues), len(dna2_residues)),
        dtype=float
    )

    # Convert global residue indices to matrix indices
    tail1_index = {
        res.resindex: i for i, res in enumerate(tail1_residues)
    }
    tail2_index = {
        res.resindex: i for i, res in enumerate(tail2_residues)
    }
    dna1_index = {
        res.resindex: i for i, res in enumerate(dna1_residues)
    }
    dna2_index = {
        res.resindex: i for i, res in enumerate(dna2_residues)
    }

    n_frames = 0

    for ts in u.trajectory:
        n_frames += 1

        # One Boolean residue-nucleotide contact map per frame
        frame_map1 = np.zeros_like(contact_map1, dtype=bool)
        frame_map2 = np.zeros_like(contact_map2, dtype=bool)

        # H3 tail 1-DNA end 1 atom pairs within cutoff
        atom_pairs1 = capped_distance(
            h3_tail1.positions,
            dna_end1.positions,
            max_cutoff=cutoff,
            box=ts.dimensions,
            return_distances=False
        )

        for tail_atom_index, dna_atom_index in atom_pairs1:
            tail_resindex = h3_tail1[tail_atom_index].resindex
            dna_resindex = dna_end1[dna_atom_index].resindex

            frame_map1[
                tail1_index[tail_resindex],
                dna1_index[dna_resindex]
            ] = True

        # H3 tail 2-DNA end 2 atom pairs within cutoff
        atom_pairs2 = capped_distance(
            h3_tail2.positions,
            dna_end2.positions,
            max_cutoff=cutoff,
            box=ts.dimensions,
            return_distances=False
        )

        for tail_atom_index, dna_atom_index in atom_pairs2:
            tail_resindex = h3_tail2[tail_atom_index].resindex
            dna_resindex = dna_end2[dna_atom_index].resindex

            frame_map2[
                tail2_index[tail_resindex],
                dna2_index[dna_resindex]
            ] = True

        # Count each residue-nucleotide contact only once per frame
        contact_map1 += frame_map1
        contact_map2 += frame_map2

    # Convert frame counts to contact probabilities between 0 and 1
    contact_map1 /= n_frames
    contact_map2 /= n_frames

    # Axis labels
    tail1_labels = [
        f"{res.resname}{res.resid}" for res in tail1_residues
    ]
    tail2_labels = [
        f"{res.resname}{res.resid}" for res in tail2_residues
    ]

    dna1_labels = [
        f"{res.atoms[0].chainID}:{res.resname}{res.resid}"
        for res in dna1_residues
    ]
    dna2_labels = [
        f"{res.atoms[0].chainID}:{res.resname}{res.resid}"
        for res in dna2_residues
    ]

    # Save maps with labels
    file1 = f"h3tail1_dnaend1_contact_map_umb{window}.dat"
    with open(file1, "w") as output:
        output.write(
            "# Heavy-atom contact probability; cutoff = 4.5 A\n"
        )
        output.write(
            "# H3_residue\t" + "\t".join(dna1_labels) + "\n"
        )

        for label, row in zip(tail1_labels, contact_map1):
            values = "\t".join(f"{value:.6f}" for value in row)
            output.write(f"{label}\t{values}\n")

    file2 = f"h3tail2_dnaend2_contact_map_umb{window}.dat"
    with open(file2, "w") as output:
        output.write(
            "# Heavy-atom contact probability; cutoff = 4.5 A\n"
        )
        output.write(
            "# H3_residue\t" + "\t".join(dna2_labels) + "\n"
        )

        for label, row in zip(tail2_labels, contact_map2):
            values = "\t".join(f"{value:.6f}" for value in row)
            output.write(f"{label}\t{values}\n")

    # Plot both contact maps
    fig, axes = plt.subplots(
        2, 1,
        figsize=(15, 12),
        constrained_layout=True
    )

    im1 = axes[0].imshow(
        contact_map1,
        origin="lower",
        aspect="auto",
        cmap="Oranges",
        vmin=0,
        vmax=1
    )

    axes[0].set_title(
        f"Umbrella {window}: H3 tail 1–DNA end 1"
    )
    axes[0].set_ylabel("H3 tail residue")
    axes[0].set_xticks(np.arange(len(dna1_labels)))
    axes[0].set_xticklabels(
        dna1_labels,
        rotation=90,
        fontsize=6
    )
    axes[0].set_yticks(np.arange(len(tail1_labels)))
    axes[0].set_yticklabels(tail1_labels, fontsize=7)

    im2 = axes[1].imshow(
        contact_map2,
        origin="lower",
        aspect="auto",
        cmap="Oranges",
        vmin=0,
        vmax=1
    )

    axes[1].set_title(
        f"Umbrella {window}: H3 tail 2–DNA end 2"
    )
    axes[1].set_xlabel("DNA nucleotide")
    axes[1].set_ylabel("H3 tail residue")
    axes[1].set_xticks(np.arange(len(dna2_labels)))
    axes[1].set_xticklabels(
        dna2_labels,
        rotation=90,
        fontsize=6
    )
    axes[1].set_yticks(np.arange(len(tail2_labels)))
    axes[1].set_yticklabels(tail2_labels, fontsize=7)

    colorbar = fig.colorbar(
        im2,
        ax=axes,
        shrink=0.85,
        pad=0.02
    )
    colorbar.set_label("Contact probability")

    fig.savefig(
        f"h3_dna_contact_maps_umb{window}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)

    print(
        f"Completed umbrella {window}: "
        f"{n_frames} frames analyzed"
    )